# Project Idea
- We collect job postings from LinkedIn and at least one additional job board, then apply text mining and LLM based methods to extract skills, keywords, and trends that can be compared against candidate resumes.

Notebok Structure:

1. Introduction & Project Goal
2. Data Collection
   - LinkedIn scraping
   - Indeed scraping
3. Data Cleaning & Dataset Creation
4. Exploratory Text Analysis
5. Skill Extraction
   - Keyword based
   - LLM based
6. Comparison & Findings
7. Limitations & Ethics

## The job posting dataset was collected using the Python JobSpy library, which supports scraping multiple job boards through a unified interface. In this project, postings were gathered from LinkedIn and Indeed for the role “data scientist” in the San Francisco Bay Area. JobSpy can also be configured to search additional sources such as Glassdoor, ZipRecruiter, Google Jobs, and others, allowing the data collection layer to be extended without changing the downstream text mining pipeline.


### Phase 1: Classic NLP (baseline)
	•	Word frequency
	•	TF-IDF keywords
	•	Skill dictionary matching
	•	Skill coverage per role

Outputs:

	•	tables
	•	bar charts
	•	comparison matrices


### Phase 2: LLM powered extraction (this satisfies the course theme)

Use an LLM to extract structured skills from descriptions.

Example prompt:

Extract a list of technical skills, tools, and programming languages from the following job description. Return a comma separated list.

Then compare:

	•	TF-IDF keywords vs LLM extracted skills
	•	Precision differences
	•	Noise reduction


## Test area - Run after imports to ensure dependencies are installed

In [ ]:
from jobspy import scrape_jobs
# Test run
# Linkedin
jobs = scrape_jobs(
    site_name=["linkedin"],
    search_term="software engineer",
    location="San Francisco, CA",
    results_wanted=10,
    linkedin_fetch_description=False,
    verbose=2,
)

print("rows:", len(jobs))
print(jobs[["title","company","location","job_url"]].head())

2026-01-27 19:20:27,214 - INFO - JobSpy:LinkedIn - search page: 1 / 1
2026-01-27 19:20:28,116 - INFO - JobSpy:Linkedin - finished scraping


rows: 10
                                  title                         company  \
0  Software Engineer, Autonomy-New Grad                            Nuro   
1                   Software Engineer I  Sony Interactive Entertainment   
2          Software Engineer - Frontend                        LinkedIn   
3                   Software Engineer 1                          Intuit   
4            Software Engineer, Backend                          Tinder   

            location                                        job_url  
0  Mountain View, CA  https://www.linkedin.com/jobs/view/4327790555  
1      San Mateo, CA  https://www.linkedin.com/jobs/view/4355710101  
2  Mountain View, CA  https://www.linkedin.com/jobs/view/4365237330  
3  Mountain View, CA  https://www.linkedin.com/jobs/view/4363662440  
4      Palo Alto, CA  https://www.linkedin.com/jobs/view/4328991042  


In [ ]:
jobs.to_csv("jobs.csv", index=False)

In [ ]:
from jobspy import scrape_jobs
import pandas as pd

jobs = scrape_jobs(
    site_name=["linkedin"],
    search_term="data scientist",          # change per teammate
    location="San Francisco Bay Area",     # broaden a bit
    results_wanted=50,                     # start 50, scale later
    linkedin_fetch_description=True,       # KEY for keyword matching
    verbose=1,
)

print("rows:", len(jobs))
jobs.head(2)

rows: 50


,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,li-4365449132,linkedin,https://www.linkedin.com/jobs/view/4365449132,https://tnl2.jometer.com/v2/job?jz=5wqzs841272...,Data Scientist,Uber,"San Francisco, CA",2026-01-23,fulltime,NaN,...,None,None,None,None,None,None,None,None,None,None
1,li-4365436354,linkedin,https://www.linkedin.com/jobs/view/4365436354,NaN,Data Scientist,Uber,"Sunnyvale, CA",2026-01-23,fulltime,NaN,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
jobs.to_csv("jobs.csv", index=False)

In [ ]:
import pandas as pd
import re

ANALYSIS_COLS = [
    "site",
    "search_term",      # we’ll add this
    "title",
    "company",
    "location",
    "date_posted",
    "job_type",
    "description",
    "job_url"
]

# add search_term so teammates can merge later
jobs["search_term"] = "data scientist"

df = jobs[[c for c in ANALYSIS_COLS if c in jobs.columns]].copy()

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["description_clean"] = df["description"].apply(clean_text)
df["title_clean"] = df["title"].apply(clean_text)

# Combined text field for NLP
df["text"] = (df["title_clean"] + " " + df["description_clean"]).str.lower()

df.head(2)

,site,search_term,title,company,location,date_posted,job_type,description,job_url,description_clean,title_clean,text
0,linkedin,data scientist,Data Scientist,Uber,"San Francisco, CA",2026-01-23,fulltime,**About The Role**\nWe are looking for an exce...,https://www.linkedin.com/jobs/view/4365449132,**About The Role** We are looking for an excep...,Data Scientist,data scientist **about the role** we are looki...
1,linkedin,data scientist,Data Scientist,Uber,"Sunnyvale, CA",2026-01-23,fulltime,**About The Role**\nWe are looking for an exce...,https://www.linkedin.com/jobs/view/4365436354,**About The Role** We are looking for an excep...,Data Scientist,data scientist **about the role** we are looki...


# Imports and a shared standardizer


In [1]:
# 1) Install JobSpy without deps
!pip -q install -U --no-cache-dir python-jobspy --no-deps


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [2]:
# 2) Pin the exact / compatible versions JobSpy 1.1.82 needs
!pip -q install --no-cache-dir \
  "numpy==1.26.3" \
  "markdownify==0.13.1" \
  "regex>=2024.4.28,<2025.0.0" \
  tls-client requests beautifulsoup4 pydantic typing-extensions pandas


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [3]:
# Query and transform the data
!pip -q install duckdb

### Sanity check that things are imported properly

In [4]:
import numpy as np, pandas as pd, duckdb
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("duckdb", duckdb.__version__)

from jobspy import scrape_jobs
print("jobspy import ok")

numpy: 1.26.3
pandas: 2.3.3
duckdb 1.4.4
jobspy import ok


In [5]:
from datetime import datetime
from pathlib import Path

TODAY = datetime.now().strftime("%Y-%m-%d")

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [6]:
from jobspy import scrape_jobs
import pandas as pd
import re

SEARCH_TERM = "data scientist"
LOCATION = "San Francisco Bay Area"
RESULTS_PER_SITE = 100  # adjust as needed

# Standard schema we want across sources
FINAL_COLS = [
    "source", "search_term",
    "title", "company", "location", "date_posted", "job_type",
    "description", "job_url"
]

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def standardize_jobs(jobs: pd.DataFrame, source: str, search_term: str) -> pd.DataFrame:
    df = jobs.copy()
    df["source"] = source
    df["search_term"] = search_term

    # Ensure expected columns exist
    for c in ["title", "company", "location", "date_posted", "job_type", "description", "job_url"]:
        if c not in df.columns:
            df[c] = ""

    df["title"] = df["title"].apply(clean_text)
    df["description"] = df["description"].apply(clean_text)
    df["company"] = df["company"].apply(clean_text)
    df["location"] = df["location"].apply(clean_text)

    # Create a combined text field for NLP
    df["text"] = (df["title"] + " " + df["description"]).str.lower().apply(clean_text)

    # Keep only the columns we want (plus text)
    keep = FINAL_COLS + ["text"]
    return df[keep]

## Scrape LinkedIn and save

In [10]:
from datetime import datetime
jobs_linkedin_raw = scrape_jobs(
    site_name=["linkedin"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=RESULTS_PER_SITE,
    linkedin_fetch_description=True,  # important for keyword extraction
    verbose=1,
)

df_linkedin = standardize_jobs(jobs_linkedin_raw, source="linkedin", search_term=SEARCH_TERM)
print("LinkedIn rows:", len(df_linkedin))

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"jobs_linkedin_{timestamp}.csv"

out_path = DATA_RAW / filename
df_linkedin.to_csv(out_path, index=False)

print(f"Saved file: {out_path.resolve()}")
df_linkedin.head(3)

LinkedIn rows: 100
Saved file: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_linkedin_20260202_183552.csv


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text
0,linkedin,data scientist,Data Scientist Intern,Zoox,"Foster City, CA",2026-02-01,internship,Zoox’s internship program provides hands\-on e...,https://www.linkedin.com/jobs/view/4335042741,data scientist intern zoox’s internship progra...
1,linkedin,data scientist,Senior Data Scientist,Roku,"San Jose, CA",2026-02-01,fulltime,Teamwork makes the stream work. **Roku is chan...,https://www.linkedin.com/jobs/view/4313070387,senior data scientist teamwork makes the strea...
2,linkedin,data scientist,"Machine Learning Engineer, Autolabeling (Inter...",Woven by Toyota,"Palo Alto, CA",2026-01-31,internship,Woven by Toyota is enabling Toyota’s once\-in\...,https://www.linkedin.com/jobs/view/4352374154,"machine learning engineer, autolabeling (inter..."


## Scrape Indeed and save

In [11]:
jobs_indeed_raw = scrape_jobs(
    site_name=["indeed"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=RESULTS_PER_SITE,
    country_indeed="USA",
    verbose=1,
)

# Indeed descriptions are sometimes present, sometimes not.
# You can still do useful keyword analysis with title + whatever description you get.
df_indeed = standardize_jobs(jobs_indeed_raw, source="indeed", search_term=SEARCH_TERM)
print("Indeed rows:", len(df_indeed))

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"jobs_indeed_{timestamp}.csv"

out_path = DATA_RAW / filename
df_indeed.to_csv(out_path, index=False)

print(f"Saved file: {out_path.resolve()}")
df_indeed.head(3)

Indeed rows: 100
Saved file: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/raw/jobs_indeed_20260202_183553.csv


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text
0,indeed,data scientist,Lead Applied Data Scientist - Search and Ranki...,Target,"Sunnyvale, CA, US",2026-02-01,fulltime,"The pay range is $132,000\.00 \- $286,000\.00 ...",https://www.indeed.com/viewjob?jk=26b7735e3a29...,lead applied data scientist - search and ranki...
1,indeed,data scientist,Lead Data Scientist - Last Mile Supply Chain O...,Target,"Sunnyvale, CA, US",2026-02-01,fulltime,"The pay range is $132,000\.00 \- $286,000\.00 ...",https://www.indeed.com/viewjob?jk=3b59a22c76a0...,lead data scientist - last mile supply chain o...
2,indeed,data scientist,Lead Machine Learning Engineer - Personalizati...,Target,"Sunnyvale, CA, US",2026-02-01,fulltime,"The pay range is $132,000\.00 \- $286,000\.00 ...",https://www.indeed.com/viewjob?jk=72ce814b0359...,lead machine learning engineer - personalizati...


# SQL to combine them into one dataset

In [13]:
import duckdb
from pathlib import Path

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

query = f"""
CREATE OR REPLACE TABLE combined_jobs AS
SELECT * FROM read_csv_auto('{(DATA_RAW / "jobs_linkedin_*.csv").as_posix()}')
UNION ALL
SELECT * FROM read_csv_auto('{(DATA_RAW / "jobs_indeed_*.csv").as_posix()}');
"""
con.execute(query)

combined = con.execute("SELECT * FROM combined_jobs").df()
print("Combined rows:", len(combined))
combined.head(5)

Combined rows: 200
Saved: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/processed/jobs_combined.csv


In [ ]:
# Save the dataset as jobs_combined
out_combined = DATA_PROCESSED / "jobs_combined.csv"
combined.to_csv(out_combined, index=False)
print("Saved:", out_combined.resolve())

## remove duplicates in SQL

In [14]:
dedup_query = """
CREATE OR REPLACE TABLE combined_jobs_dedup AS
SELECT *
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY
        COALESCE(job_url, ''),
        COALESCE(title, ''),
        COALESCE(company, '')
      ORDER BY date_posted DESC NULLS LAST
    ) AS rn
  FROM combined_jobs
)
WHERE rn = 1;
"""
con.execute(dedup_query)

combined_dedup = con.execute("SELECT * FROM combined_jobs_dedup").df()
print("Dedup rows:", len(combined_dedup))
combined_dedup.head(5)

Dedup rows: 200


,source,search_term,title,company,location,date_posted,job_type,description,job_url,text,rn
0,indeed,data scientist,"Applied Scientist, Last Mile Delivery Automation",Amazon.com,"Santa Clara, CA, US",2026-01-04,fulltime,**DESCRIPTION** --------------- As an Applied ...,https://www.indeed.com/viewjob?jk=16635ed300fb...,"applied scientist, last mile delivery automati...",1
1,indeed,data scientist,"Applied Scientist, Delivery Foundation Model",Amazon.com,"Santa Clara, CA, US",2025-12-18,fulltime,**DESCRIPTION** --------------- Join the next ...,https://www.indeed.com/viewjob?jk=35b80b2a7d03...,"applied scientist, delivery foundation model *...",1
2,indeed,data scientist,"Principal Applied Scientist, AGI Foundations, ...",Amazon.com,"Sunnyvale, CA, US",2025-07-18,fulltime,**DESCRIPTION** --------------- As a Principal...,https://www.indeed.com/viewjob?jk=495994bc3b54...,"principal applied scientist, agi foundations, ...",1
3,indeed,data scientist,"Applied Scientist II, Amazon Stores Finance Sc...",Amazon.com,"Cupertino, CA, US",2026-01-06,fulltime,**DESCRIPTION** --------------- Amazon Stores ...,https://www.indeed.com/viewjob?jk=861ab34c6cbc...,"applied scientist ii, amazon stores finance sc...",1
4,indeed,data scientist,"Applied Scientist, Safety ML",Amazon.com,"San Francisco, CA, US",2026-01-15,fulltime,**DESCRIPTION** --------------- If you are int...,https://www.indeed.com/viewjob?jk=fd01313b594c...,"applied scientist, safety ml **description** -...",1


In [15]:
# saved the processed data as jobs_combined_dedup
out_dedup = DATA_PROCESSED / "jobs_combined_dedup.csv"
combined_dedup.to_csv(out_dedup, index=False)
print("Saved:", out_dedup.resolve())

Saved: /Users/Jordan/Documents/USD/Spring 2026/ADS 509/Final Project/data/processed/jobs_combined_dedup.csv


# Dataset 3 - Optional for other sites